In [10]:
# Update package manager
!apt-get update && apt-get install -y wget xz-utils libegl1 libgl1-mesa-glx libglib2.0-0 ffmpeg
!apt-get update && apt-get install -y liblog4cplus-dev
!apt-get update && apt-get install -y ffmpeg

!apt-get update && apt-get install -y \
    libgl1 \
    libx11-6 \
    libxxf86vm1 \
    libxcursor1 \
    libxi6 \
    libxrandr2 \
    libxinerama1 \
    libxkbcommon0 \
    libsm6 \
    libice6

# Download official Blender 4.0 (or whichever version fits your pipeline)
!wget https://download.blender.org/release/Blender4.0/blender-4.0.2-linux-x64.tar.xz

# Extract it
!tar -xf blender-4.0.2-linux-x64.tar.xz
!mv blender-4.0.2-linux-x64 /opt/blender

# Verify it works headlessly
!/opt/blender/blender --background --version

Hit:1 http://archive.ubuntu.com/ubuntu noble InRelease
Hit:2 http://security.ubuntu.com/ubuntu noble-security InRelease
Hit:3 http://archive.ubuntu.com/ubuntu noble-updates InRelease
Hit:4 http://archive.ubuntu.com/ubuntu noble-backports InRelease
Reading package lists... Done
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
Note, selecting 'libglib2.0-0t64' instead of 'libglib2.0-0'
Package libgl1-mesa-glx is not available, but is referred to by another package.
This may mean that the package is missing, has been obsoleted, or
is only available from another source

E: Package 'libgl1-mesa-glx' has no installation candidate
Hit:1 http://security.ubuntu.com/ubuntu noble-security InRelease
Hit:2 http://archive.ubuntu.com/ubuntu noble InRelease
Hit:3 http://archive.ubuntu.com/ubuntu noble-updates InRelease
Hit:4 http://archive.ubuntu.com/ubuntu noble-backports InRelease
Reading package lists... Done
Reading package lists... Done
Building dep

Hit:1 http://security.ubuntu.com/ubuntu noble-security InRelease
Hit:2 http://archive.ubuntu.com/ubuntu noble InRelease
Hit:3 http://archive.ubuntu.com/ubuntu noble-updates InRelease
Hit:4 http://archive.ubuntu.com/ubuntu noble-backports InRelease
Reading package lists... Done
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  adwaita-icon-theme at-spi2-common at-spi2-core dbus dbus-bin dbus-daemon
  dbus-session-bus-common dbus-system-bus-common dbus-user-session
  dconf-gsettings-backend dconf-service dmsetup fontconfig fontconfig-config
  fonts-dejavu-core fonts-dejavu-mono gir1.2-girepository-2.0 gir1.2-glib-2.0
  gsettings-desktop-schemas gtk-update-icon-cache hicolor-icon-theme
  humanity-icon-theme i965-va-driver intel-media-va-driver libaacs0 libaom3
  libapparmor1 libargon2-1 libass9 libasyncns0 libatk-bridge2.0-0t64
  libatk1.0-0t64 libatspi2.0-0t64 libavahi-client3 libavahi-c

In [23]:
import numpy as np
import os
import subprocess
from tqdm import tqdm
from scipy.ndimage import gaussian_filter
from skimage import measure

# --- CONFIGURATION ---
BLENDER_PATH = "/opt/blender/blender"

# Output directories
os.makedirs("riboflavin_meshes", exist_ok=True)
os.makedirs("rendered_frames", exist_ok=True)

print(f"✅ Simulation initialized. Using Blender at: {BLENDER_PATH}")

# =====================================================================
# STEP 1: DUAL-STRUCTURE CRYSTALLIZATION SIMULATOR
# =====================================================================
def simulate_meniscus_crystallization(grid_size=128, total_frames=45):
    print(f"🧬 Modeling Dual-Structure Crystallization ({grid_size}^3 across {total_frames} frames)...")
    C = np.zeros((grid_size, grid_size, grid_size), dtype=np.float32)
    
    # Define the Dense Rim Arc (The Coffee-Ring Meniscus)
    R = grid_size * 0.45
    ring_width = grid_size * 0.04
    y, x, z = np.indices((grid_size, grid_size, grid_size))
    meniscus_mask = (x-grid_size//2)**2 + (y-grid_size//2)**2 > (R-ring_width)**2
    meniscus_mask &= (x-grid_size//2)**2 + (y-grid_size//2)**2 < (R+ring_width)**2
    meniscus_mask &= (y < grid_size * 0.85)
    C[meniscus_mask & (z > grid_size*0.35) & (z < grid_size*0.45)] = 0.15

    history = []
    for t in tqdm(range(total_frames), desc="Running Crystallization Simulation"):
        C_force = gaussian_filter(C, sigma=1.2)
        
        # Dense acicular growth along the rim boundary
        C = np.where((meniscus_mask) & (C_force > 0.1) & (C < 0.9), C + 0.3, C)
        
        # Branching internal plume kinetics dropping off the rim into the interior
        if t > 5:
            # Noise seed to grow back dendritic structures seen in Capture_1
            noise = np.random.rand(grid_size, grid_size, grid_size) > 0.990
            C = np.where(noise & (C_force > 0.04) & (C < 0.15), C + 0.35, C)

        C = np.clip(C, 0.0, 1.0)
        history.append(C.copy())
        
    return np.array(history)

simulation_data = simulate_meniscus_crystallization(grid_size=128, total_frames=45)

# =====================================================================
# STEP 2: EXPORT TO STANDARD 3D MESHES (.OBJ)
# =====================================================================
def export_mesh_frames(sim_data):
    print("📐 Converting physics data to 3D Mesh streams (.obj)...")
    mesh_paths = []
    
    for t in tqdm(range(len(sim_data)), desc="Writing Mesh Files"):
        frame_data = sim_data[t]
        if np.max(frame_data) < 0.3:
            continue
            
        # Lower marching cubes threshold to capture the lighter internal plumes
        verts, faces, normals, values = measure.marching_cubes(frame_data, level=0.3)
        
        obj_filepath = f"riboflavin_meshes/frame_{t:04d}.obj"
        with open(obj_filepath, 'w') as f:
            for v in verts:
                f.write(f"v {v[0]} {v[1]} {v[2]}\n")
            for face in faces:
                f.write(f"f {face[0]+1} {face[1]+1} {face[2]+1}\n")
                
        mesh_paths.append(obj_filepath)
    return mesh_paths

mesh_paths = export_mesh_frames(simulation_data)
# =====================================================================
# STEP 3: API-ROBUST MICROSCOPY RENDERING ENGINE
# =====================================================================
def generate_render_frame_with_bpy(obj_filepath, output_image_path):
    blender_script_content = f'''
import bpy
import os

scene = bpy.context.scene
scene.render.engine = 'CYCLES'
scene.cycles.device = 'CPU' 
scene.render.resolution_x = 1280
scene.render.resolution_y = 720
scene.cycles.samples = 16 
scene.render.filepath = "{output_image_path}"

# Clear default layout objects completely
for obj in bpy.data.objects:
    bpy.data.objects.remove(obj, do_unlink=True)

# --- 1. Microscope Background Slide (The Glass Substrate) ---
bpy.ops.mesh.primitive_plane_add(size=500, location=(64, 64, 20))
slide_plane = bpy.context.active_object
slide_plane.name = "Microscope_Slide"

slide_mat = bpy.data.materials.new(name="SlideMaterial")
slide_mat.use_nodes = True
# Light greyish-blue backlit coloration matching microscope fields
slide_mat.node_tree.nodes.get("Principled BSDF").inputs['Base Color'].default_value = (0.88, 0.92, 0.95, 1.0)
slide_mat.node_tree.nodes.get("Principled BSDF").inputs['Roughness'].default_value = 1.0
slide_plane.data.materials.append(slide_mat)

# --- 2. Camera Setup ---
camera = bpy.data.cameras.new("MicroscopeCamera")
camera_obj = bpy.data.objects.new("MicroscopeCamera", camera)
scene.collection.objects.link(camera_obj)
scene.camera = camera_obj
camera_obj.location = (64, 64, 215)  # Positioned above looking down at the 128x128 grid
camera_obj.rotation_euler = (0, 0, 0)

# --- 3. Illumination Environment ---
light_data = bpy.data.lights.new("MainBright", 'AREA')
light_obj = bpy.data.objects.new("MainBright", light_data)
scene.collection.objects.link(light_obj)
light_obj.location = (64, 64, 190)
light_obj.data.energy = 80000  
light_obj.data.size = 200

scene.world.use_nodes = True
scene.world.node_tree.nodes.get('Background').inputs['Color'].default_value = (0.9, 0.93, 0.96, 1.0)
scene.world.node_tree.nodes.get('Background').inputs['Strength'].default_value = 1.5

# --- 4. Ingest & Manipulate Crystal Mesh via Data Properties ---
bpy.ops.wm.obj_import(filepath="{obj_filepath}")

# Target the imported mesh without relying on selection states
crystal_obj = None
for obj in scene.objects:
    if obj.type == 'MESH' and obj != slide_plane:
        crystal_obj = obj
        break

if crystal_obj:
    # CONTEXT-SAFE THICKNESS FIX:
    # Use explicit modifier assignment rather than calling bpy.ops viewport tools
    solid_mod = crystal_obj.modifiers.new(name="CrystalThickness", type='SOLIDIFY')
    solid_mod.thickness = 3.5  # Thicker expansion so it's fully visible to light rays
    solid_mod.offset = 0.0
    
    # Smooth out pixelated voxel step lines to look like crystalline aggregates
    smooth_mod = crystal_obj.modifiers.new(name="OrganicSmooth", type='SMOOTH')
    smooth_mod.factor = 0.6
    smooth_mod.iterations = 3

    # --- 5. High-Contrast Dense Surface Material ---
    # Bypasses translucent refraction bugs by blending opaque amber and glowing yellow
    mat = bpy.data.materials.new(name="RiboflavinMaterial")
    mat.use_nodes = True
    nodes = mat.node_tree.nodes
    links = mat.node_tree.links
    nodes.clear()
    
    # Dark brown-amber core shader (representing dense needle clustering)
    node_core = nodes.new('ShaderNodeBsdfPrincipled')
    node_core.inputs['Base Color'].default_value = (0.08, 0.04, 0.005, 1.0) 
    node_core.inputs['Roughness'].default_value = 0.7
    
    # Bright orange-yellow fringe shader (representing thin internal plumes)
    node_edge = nodes.new('ShaderNodeBsdfPrincipled')
    node_edge.inputs['Base Color'].default_value = (1.0, 0.55, 0.02, 1.0)
    node_edge.inputs['Roughness'].default_value = 0.4
    
    # Blend shaders using viewing angles to expose structural details
    node_fresnel = nodes.new('ShaderNodeFresnel')
    node_fresnel.inputs['IOR'].default_value = 1.45
    
    node_mix = nodes.new('ShaderNodeMixShader')
    node_output = nodes.new('ShaderNodeOutputMaterial')
    
    # Link network logic
    links.new(node_fresnel.outputs['Fac'], node_mix.inputs['Fac'])
    links.new(node_core.outputs['BSDF'], node_mix.inputs[1])
    links.new(node_edge.outputs['BSDF'], node_mix.inputs[2])
    links.new(node_mix.outputs['BSDF'], node_output.inputs['Surface'])

    crystal_obj.data.materials.append(mat)

# Render frame headless
bpy.ops.render.render(write_still=True)
'''
    
    temp_script_path = "temp_render_script.py"
    with open(temp_script_path, "w") as f:
        f.write(blender_script_content)
        
    cmd = [BLENDER_PATH, "--background", "--python", temp_script_path]
    result = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    
    if not os.path.exists(output_image_path):
        print(f"\n❌ Rendering failed for {os.path.basename(obj_filepath)}. Blender internal error:")
        print(result.stderr if result.stderr else result.stdout[-400:])
        return False
    return True
# =====================================================================
# STEP 4: AUTOMATED VIDEO COMPILER
# =====================================================================
def compile_macro_video_file(output_filename="macroscopic_riboflavin_growth.mp4"):
    print("🎞️ Assembling frame assets into final mp4 file...")
    if not os.listdir("rendered_frames"):
        print("⚠️ Compiling skipped: 'rendered_frames' directory is empty.")
        return
        
    try:
        cmd = [
            "ffmpeg", "-y", "-r", "15", "-i", "rendered_frames/frame_%04d.png",
            "-c:v", "libx264", "-pix_fmt", "yuv420p", "-crf", "20", output_filename
        ]
        subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, check=True)
        print(f"🎉 SUCCESS! Video saved to your Docker workspace: {os.path.abspath(output_filename)}")
    except subprocess.CalledProcessError as e:
        print("⚠️ FFmpeg execution failed. Error log:")
        print(e.stderr.decode())

compile_macro_video_file()

✅ Simulation initialized. Using Blender at: /opt/blender/blender
🧬 Modeling Dual-Structure Crystallization (128^3 across 45 frames)...


Running Crystallization Simulation: 100%|███████████████████████████████████████████| 45/45 [00:01<00:00, 24.90it/s]


📐 Converting physics data to 3D Mesh streams (.obj)...


Writing Mesh Files: 100%|███████████████████████████████████████████████████████████| 45/45 [00:09<00:00,  4.86it/s]


🎞️ Assembling frame assets into final mp4 file...
🎉 SUCCESS! Video saved to your Docker workspace: /workspace/macroscopic_riboflavin_growth.mp4
